# DUCX Colab Quickstart

This notebook launches the DUCX paper workflow for tool-using chest X-ray agents.

Set the form fields, run the cells top to bottom, and toggle `RUN_AGENT` or `RUN_ANALYSIS` when your data, images, and LLM endpoint are ready.

## 1. Clone And Install

When opened in Colab from GitHub, the notebook does not automatically clone the repository. Set `REPO_URL` to the public repository URL before running this cell in Colab. Local runs from the repository root skip cloning and package installation by default.

In [ ]:
# @title Repository setup
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
INSTALL_PACKAGE = "auto"  # @param ["auto", "yes", "no"]

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    if not REPO_URL:
        raise ValueError("Set REPO_URL to the public DUCX GitHub repository URL before running in Colab.")
    repo_name = Path(REPO_URL.rstrip('/').removesuffix('.git')).name or "DUCX"
    repo_dir = Path('/content') / repo_name
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
else:
    if not Path('pyproject.toml').exists():
        raise RuntimeError('Run this notebook from the DUCX repository root, or open it in Colab and set REPO_URL.')

should_install = INSTALL_PACKAGE == 'yes' or (INSTALL_PACKAGE == 'auto' and IN_COLAB)
if should_install:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
else:
    print('Package install skipped. Set INSTALL_PACKAGE="yes" to force editable install.')

print('Working directory:', Path.cwd())
print('Python:', sys.executable)

## 2. Configure Data

MIMIC-FairnessVQA is provided by the DUCX paper release. ChestAgentBench / EuroRAD metadata is provided by MedRAX: https://github.com/bowang-lab/MedRAX/tree/main/data.

Place data and images at the configured paths before running an actual agent job.

In [ ]:
# @title Data paths
from pathlib import Path

DATASET = "mimic"  # @param ["mimic", "chestagentbench"]
MIMIC_QUESTION_FILE = "data/mimic/medrax_input_all_2000.jsonl"  # @param {type:"string"}
MIMIC_CASE_META = "data/mimic/mimic_sample_400.csv"  # @param {type:"string"}
CHESTAGENTBENCH_QUESTION_FILE = "data/chestagentbench/metadata.jsonl"  # @param {type:"string"}
CHESTAGENTBENCH_CASE_META = "data/eurorad_metadata.json"  # @param {type:"string"}

for directory in ['data/chestagentbench', 'data/mimic', 'figures', 'logs', 'model-weights', 'temp']:
    Path(directory).mkdir(parents=True, exist_ok=True)

if DATASET == 'mimic':
    DATA_FILE = MIMIC_QUESTION_FILE
    META_CASE_PATH = MIMIC_CASE_META
    DEFAULT_LOG_PREFIX = 'qwen3vl8b-vllm-mimic'
    DEFAULT_ANALYSIS_DIR = 'logs/mimic/analysis/qwen3vl8b-vllm/fairness_posthoc'
else:
    DATA_FILE = CHESTAGENTBENCH_QUESTION_FILE
    META_CASE_PATH = CHESTAGENTBENCH_CASE_META
    DEFAULT_LOG_PREFIX = 'qwen3vl8b-vllm'
    DEFAULT_ANALYSIS_DIR = 'logs/chexagentbench/analysis/qwen3vl8b-vllm/fairness_posthoc'

print('DATA_FILE =', DATA_FILE, '| exists:', Path(DATA_FILE).exists())
print('META_CASE_PATH =', META_CASE_PATH, '| exists:', Path(META_CASE_PATH).exists())
print()
print('See data/README.md for download/provenance details.')

## 3. Configure LLM Endpoint

For local vLLM or another OpenAI-compatible server, set `OPENAI_BASE_URL`, `OPENAI_API_KEY`, and `OPENAI_MODEL`. For native Gemini tool calling, set `USE_GEMINI_NATIVE=True` and provide `GEMINI_API_KEY` in the environment.

In [ ]:
# @title LLM and run configuration
import os

OPENAI_BASE_URL = "http://localhost:8000/v1"  # @param {type:"string"}
OPENAI_API_KEY = "EMPTY"  # @param {type:"string"}
OPENAI_MODEL = "qwen3-vl-8b"  # @param {type:"string"}
USE_GEMINI_NATIVE = False  # @param {type:"boolean"}
DEVICE = "cuda"  # @param ["cuda", "cpu"]
MAX_CASES = 5  # @param {type:"integer"}
LOG_PREFIX = ""  # @param {type:"string"}

os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['OPENAI_MODEL'] = OPENAI_MODEL

LOG_PREFIX = LOG_PREFIX or DEFAULT_LOG_PREFIX
print('Model:', OPENAI_MODEL)
print('Log prefix:', LOG_PREFIX)
print('Dataset file:', DATA_FILE)

## 4. Check Launch Command

This command is executed in `--dry-run` mode so users can verify the final configuration before starting a model job.

In [ ]:
# @title Dry-run launch command
import json
import subprocess
import sys

cmd = [
    sys.executable,
    'launch_over_chexbench.py',
    '--model', OPENAI_MODEL,
    '--model-dir', './model-weights',
    '--temp-dir', './temp',
    '--data-file', DATA_FILE,
    '--device', DEVICE,
    '--log-prefix', LOG_PREFIX,
    '--llm-parse',
    '--dry-run',
]
if USE_GEMINI_NATIVE:
    cmd.append('--gemini-native')
if MAX_CASES and MAX_CASES > 0:
    cmd.extend(['--max-cases', str(MAX_CASES)])

print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 5. Run Agent Evaluation

Set `RUN_AGENT=True` after the data files, image paths, endpoint, and model/tool weights are ready. The cell runs the same command as above without `--dry-run`.

In [ ]:
# @title Launch agent evaluation
RUN_AGENT = False  # @param {type:"boolean"}

if not RUN_AGENT:
    print('Set RUN_AGENT=True to start evaluation.')
else:
    data_path = Path(DATA_FILE)
    if not data_path.exists():
        raise FileNotFoundError(f'Data file not found: {data_path}')
    run_cmd = [x for x in cmd if x != '--dry-run']
    print(' '.join(run_cmd))
    subprocess.run(run_cmd, check=True)

## 6. Run DUCX Fairness Analysis

After an agent run finishes, set `LOG_PATH` to the generated JSON log. Enable the LLM judge only when your judge API key is configured.

In [ ]:
# @title Launch DUCX analysis
RUN_ANALYSIS = False  # @param {type:"boolean"}
LOG_PATH = ""  # @param {type:"string"}
OUT_DIR = ""  # @param {type:"string"}
ENABLE_LLM_JUDGE = False  # @param {type:"boolean"}
JUDGE_MODEL = "deepseek-chat"  # @param {type:"string"}
JUDGE_BASE_URL = "https://api.deepseek.com/v1"  # @param {type:"string"}
JUDGE_API_KEY_ENV = "DEEPSEEK_API_KEY"  # @param {type:"string"}

if not RUN_ANALYSIS:
    print('Set RUN_ANALYSIS=True after LOG_PATH points to a completed agent log.')
else:
    if not LOG_PATH:
        raise ValueError('Set LOG_PATH to a completed agent log file.')
    analysis_cmd = [
        sys.executable,
        'analysis/fairness_posthoc.py',
        '--log-path', LOG_PATH,
        '--meta-q-path', DATA_FILE,
        '--meta-case-path', META_CASE_PATH,
        '--out-dir', OUT_DIR or DEFAULT_ANALYSIS_DIR,
    ]
    if ENABLE_LLM_JUDGE:
        analysis_cmd.extend([
            '--enable-llm-judge',
            '--judge-model', JUDGE_MODEL,
            '--judge-base-url', JUDGE_BASE_URL,
            '--judge-api-key-env', JUDGE_API_KEY_ENV,
        ])
    print(' '.join(analysis_cmd))
    subprocess.run(analysis_cmd, check=True)